In [1]:
import fsspec
import xarray as xr
import scipy.spatial
import numpy as np
import pandas as pd 
import os
import argparse
from datetime import date
import datetime
from calculations.calculations import vapor_pressure
from calculations.calculations import wind_tot
from calculations.calculations import rel_hum
import regionmask
import geopandas as gpd
import scipy.stats as stats
from ERA5_functions import *

#### Tier 4 - total precip
On halt using ERA5 precip --> Potential issue being grid size too big to accurately calculate extreme precip 
(see pre = precip.where(precip > 0, other=np.nan)
pre.resample(time='1D').sum().where(pre > 0.05, other=np.nan).sum().values)))
array(0.00016407, dtype=float32) (m/day -> no days exceeded the 50mm.day threshold at all locations throughout all years)


Future work: critical impact rainfall threshold (https://handbook.climaax.eu/notebooks/workflows/HEAVY_RAINFALL/01_Extreme_precipitation/Extreme_precipitation_criticalthresholds.html)

dailystats 
daily precip rate 24-hourly) hourly 
~~hours with over 50cm/day~~
max precip rate in a day  (m/hour)

monthlystats 
TOTAL_MONTHLY_PRECI
DAYS_WITH_PRECIP
~~sum of days over 50cm/day~~
~~sum of hours over 50cm/day~~
Max precip (m/day)

yearlystats 
TOTAL_YEARLY_PRECI
DAYS_WITH_PRECIP
~~sum of days over 50cm/day~~
~~sum of hours over 50cm/day~~
MAX PRECIP RATE (m/day)

summary 
average yearly precip 
average days with precip 
total days over 50cm/day 
total hours over 50cm/day 
max precip rate (cm/day)


In [2]:
ds = xr.open_mfdataset('/data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc')
precip = ds.precip.sel(time=ds.time.dt.year <= 2024)

fpath = '/data/keeling/a/rytam2/a/iema_output/arcgis_toprocess/' #for output to arcgis

In [6]:
precip = ds.precip.sel(time=ds.time.dt.year <= 2024)*1e-3
## tot precip in each month each year
preci = precip.where(precip > 0, other=np.nan) #
preci += 1e-10

## daily precip rate (m/24h)
dailypreci = preci.resample(time='1D').sum() #m/day

## days with over 50mm/day
preci.where(preci > 0.5, other=np.nan).groupby(['time']).count().
## max precip rate in a day  (hourly)


In [66]:
pre = precip.where(precip > 0, other=np.nan)

np.unique(pre.resample(time='1D').where(pre > 0.05, other=np.nan).sum().values)

AlignmentError: cannot align objects with join='exact' where index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',)

In [72]:
pre.resample(time='1D').sum().where(pre > 0.05, other=np.nan).sum().values)

array(0.00016407, dtype=float32)

In [8]:
preci_dcount_monthly = preci.where(preci > 0.5, other=np.nan).groupby(['time.year',"time.month"]).count()
grouped_dcount_monthly = preci_dcount_monthly.rename('DAYS_WITH_PRECIP').stack(time=('year', 'month')) #days 

<xarray.DataArray 'precip' (time: 3288, lat: 29, lon: 27)> Size: 10MB
dask.array<add, shape=(3288, 29, 27), dtype=float32, chunksize=(3288, 29, 27), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * time     (time) datetime64[ns] 26kB 2016-01-01 2016-01-02 ... 2024-12-31

In [7]:

grouped_monthly = preci.groupby(['time.year', 'time.month']).sum(dim='time').rename('TOTAL_MONTHLY_PRECI').stack(time=('year', 'month')) # in meters/month

## tot days with rain each month each year
preci_dcount_monthly = preci.where(preci > 1e-3, other=np.nan).groupby(['time.year',"time.month"]).count()
grouped_dcount_monthly = preci_dcount_monthly.rename('DAYS_WITH_PRECIP').stack(time=('year', 'month')) #days 

## days with over 50cm/day 


monthly_compiled = xr.merge([grouped_monthly,grouped_dcount_monthly]).reset_index('time')
monthly_compiled['time']=pd.date_range(start='2015-01-01', periods=9*12, freq='MS')

# monthly_compiled.to_netcdf(fpath+'monthlystats_'+'totprecip'+'_by_county_2016-2024'+'.nc')

In [8]:
monthly_compiled

<xarray.Dataset> Size: 1MB
Dimensions:              (lat: 29, lon: 27, time: 108)
Coordinates:
  * lat                  (lat) float32 116B 43.25 43.0 42.75 ... 36.5 36.25
  * lon                  (lon) float32 108B 267.2 267.5 267.8 ... 273.5 273.8
    county               (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    year                 (time) int64 864B 2016 2016 2016 ... 2024 2024 2024
    month                (time) int64 864B 1 2 3 4 5 6 7 8 ... 6 7 8 9 10 11 12
  * time                 (time) datetime64[ns] 864B 2015-01-01 ... 2023-12-01
Data variables:
    TOTAL_MONTHLY_PRECI  (lat, lon, time) float32 338kB dask.array<chunksize=(29, 27, 108), meta=np.ndarray>
    DAYS_WITH_PRECIP     (lat, lon, time) int64 677kB dask.array<chunksize=(29, 27, 108), meta=np.ndarray>

In [ ]:
## avg monthly precip across years ?? 
## avg days with rain each month across years ?? 


## total yearly precip across years 
grouped_yearly = preci.groupby(['time.year']).sum(dim='time').rename('TOTAL_YEARLY_PRECI') # in meters/year

## tot days with rain 
preci_dcount_yearly = preci.where(preci > 1e-3, other=np.nan).groupby(['time.year']).count()
grouped_dcount_yearly = preci_dcount_yearly.rename('DAYS_WITH_PRECIP')

yearly_compiled = xr.merge([grouped_yearly,grouped_dcount_yearly])

# yearly_compiled.to_netcdf(fpath+'yearlystats_'+'totprecip'+'_by_county_2016-2024'+'.nc')

In [ ]:
grouped_yearly.isel(lat=23,lon=12).values

In [ ]:
grouped_yearly.mean(dim='year').isel(lat=23,lon=12).values

In [ ]:
### summary 
## avg yearly precip across years  
grouped_yearly = grouped_yearly.mean(dim='year').rename('AVG_YEARLY_PRECI') # in meters/month


## avg days with rain across years 


## total precip across years 
grouped_yearly = preci.groupby(['time.year']).sum(dim='time').rename('TOTAL_YEARLY_PRECI') # in meters/month


preci_dcount_yearly = preci.where(preci > 1e-3, other=np.nan).groupby(['time.year']).count()
grouped_dcount_yearly = preci_dcount_yearly.rename('DAYS_WITH_PRECIP')



grouped_hourly_smry = hourlymask.sum(dim='time').rename('TOTAL_HOURS_<-20F')#.stack(time=('year', 'month'))
grouped_daily_smry = dailymask.sum(dim='time').rename('TOTAL_DAYS_<-20F')#.stack(time=('year', 'month'))

summary_compiled = xr.merge([grouped_hourly_smry,grouped_daily_smry])
# summary_compiled.to_netcdf(fpath+'summarystats_'+'totprecip'+'_by_county_2016-2024'+'.nc')